# Rocket Ascent Optimization Explorations

This notebook explores optional follow-on analyses for the toy chemical rocket ascent optimization implemented in `rocket_trajectory.ascent_opt`. Each section builds additional insight or capability.

> NOTE: The physics model is intentionally simplified (no drag, no Earth rotation, simplified thrust & guidance). Enhancements below illustrate methodology, not operational fidelity.

In [ ]:
# 1. Setup & Imports
from __future__ import annotations
import math, json, itertools, datetime
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from rocket_trajectory.ascent_opt import (
    RocketParams, TargetOrbit, optimize_pitch, simulate_ascent, MU_EARTH, R_EARTH, G0, pitch_profile
)
plt.style.use('seaborn-v0_8-darkgrid')

RESULTS_DIR = Path('../analysis_outputs')
RESULTS_DIR.mkdir(exist_ok=True)
print('Setup complete.')

In [ ]:
# 2. Baseline Optimization Run
rp = RocketParams(thrust=2.5e6, isp=300.0, dry_mass=30_000.0, prop_mass=200_000.0)
target = TargetOrbit(altitude=200e3)
result = optimize_pitch(rp, target)
print(result)
meta = {
    'timestamp': datetime.datetime.utcnow().isoformat() + 'Z',
    'pitch_params': result.pitch_params.tolist(),
    'final_altitude_m': result.final_altitude,
    'final_mass_kg': result.final_mass,
    'delta_v_est_m_s': result.delta_v,
}
meta

In [ ]:
# 3. Inspect Raw solve_ivp Dense Output (sol.sol)
# We re-run simulate_ascent to get time/state arrays (dense evaluation inside).
tf = result.time_of_flight
t, y = simulate_ascent(rp, result.pitch_params, tf)
# Sample a finer time grid to check continuity
fine_t = np.linspace(0, tf, 2000)
# Re-evaluate dense solution by re-calling simulate_ascent (toy: just using interpolation stored as sol.sol inside function not returned)
# For strict dense solution access we would modify simulate_ascent to optionally return the raw SolveIVP object.
# For now we approximate by linear interpolation between our sampled points.
interp_r = np.interp(fine_t, t, y[0])
print('Coarse samples:', t.shape, 'Fine samples:', fine_t.shape)
print('Radius range (m):', float(y[0].min()), float(y[0].max()))

In [ ]:
# 4. Plot Time Histories (Altitude, Velocities, Mass)
altitude = y[0] - R_EARTH
fig, axs = plt.subplots(3, 1, figsize=(8, 10), sharex=True)
axs[0].plot(t, altitude/1000)
axs[0].axhline(target.altitude/1000, color='r', ls='--', label='Target')
axs[0].set_ylabel('Altitude (km)'); axs[0].legend()
axs[1].plot(t, y[1], label='v_r')
axs[1].plot(t, y[2], label='v_t')
axs[1].set_ylabel('Velocity (m/s)'); axs[1].legend()
axs[2].plot(t, y[3])
axs[2].set_ylabel('Mass (kg)'); axs[2].set_xlabel('Time (s)')
fig.tight_layout()
fig

In [ ]:
# 5. Compute Orbit Insertion Metrics (v_circ, flight path angle)
final_r = y[0,-1]; final_vr = y[1,-1]; final_vt = y[2,-1]
vcirc = math.sqrt(MU_EARTH / final_r)
flight_path_angle = math.degrees(math.atan2(final_vr, final_vt))
print({'altitude_err_m': final_r - R_EARTH - target.altitude,
       'vertical_velocity_m_s': final_vr,
       'tangential_speed_err_m_s': final_vt - vcirc,
       'flight_path_angle_deg': flight_path_angle})